# 08 — Local Gap Reconstruction

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Purpose:** improve candidate completion using local prime-gap statistics rather than global density alone.

Notebook chain:

```text
07: global density-guided reconstruction
08: local gap-guided reconstruction
```

Notebook 07 showed:

> density reconstruction can repair global count but still miss local spacing structure.

Notebook 08 adds a local gap constraint:

\[
g(x) \sim \log x
\]

Core claim:

> Density restores missing mass; local gap scoring improves structural realism.

## 0. Setup

Artifact structure:

```text
08_local_gap_reconstruction/
├── data/
├── docs/
├── figures/
└── tex/
```

Root export:

```text
08_local_gap_reconstruction_export.zip
```

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "08_local_gap_reconstruction"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]
NOTEBOOK_TITLE = "Local Gap Reconstruction"

OUT = Path(NOTEBOOK_ID)
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
FIG_DIR = OUT / "figures"
TEX_DIR = OUT / "tex"

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifact directory: {OUT.resolve()}")

## 1. Premise

Notebook 07 exposed a limitation:

\[
\text{sieve} + \text{global density} \neq \text{local prime structure}
\]

This notebook compares two reconstruction methods:

1. **Density-only reconstruction**  
   Fill missing mass to match global prime-density estimates.

2. **Local-gap reconstruction**  
   Fill missing mass using both density need and local gap consistency.

Candidate scoring:

\[
score(x)=w_d\,score_{\mathrm{density}}(x)+w_g\,score_{\mathrm{gap}}(x)+w_n\,score_{\mathrm{neighbor}}(x)
\]

## 2. Metrics

For reconstructed set \(\hat{P}_N\):

\[
precision = \frac{|\hat{P}_N \cap P_N|}{|\hat{P}_N|}
\]

\[
recovery = \frac{|\hat{P}_N \cap P_N|}{|P_N|}
\]

\[
F_1 = \frac{2\cdot precision\cdot recovery}{precision+recovery}
\]

Gap drift:

\[
drift_{\mathrm{gap}} =
\operatorname{mean}_i \frac{|g_i-\log p_i|}{\log p_i}
\]

Gap residual:

\[
r_i = g_i - \log p_i
\]

In [ ]:
# Parameters

N_MAX = 200_000
RANDOM_SEED = 9423

SCENARIOS = [
    {"name": "mixed_keep_75_noise_25", "kind": "mixed", "keep_fraction": 0.75, "noise_fraction": 0.25},
    {"name": "mixed_keep_50_noise_50", "kind": "mixed", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "mixed_keep_25_noise_100", "kind": "mixed", "keep_fraction": 0.25, "noise_fraction": 1.00},
    {"name": "biased_low_missing_high_noise", "kind": "biased_range", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "adversarial_mod6_noise_50", "kind": "adversarial_mod6", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "adversarial_mod6_noise_100", "kind": "adversarial_mod6", "keep_fraction": 0.25, "noise_fraction": 1.00},
]

Q_FILTER = int(math.sqrt(N_MAX))
COMPLETION_BIN_COUNT = 32

# Local-gap reconstruction weights.
W_DENSITY = 0.45
W_GAP = 0.40
W_NEIGHBOR = 0.15

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "SCENARIOS": SCENARIOS,
    "Q_FILTER": Q_FILTER,
    "COMPLETION_BIN_COUNT": COMPLETION_BIN_COUNT,
    "W_DENSITY": W_DENSITY,
    "W_GAP": W_GAP,
    "W_NEIGHBOR": W_NEIGHBOR,
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
}

params

## 3. Reference primes and stress-test observations

Reuse Notebook 07 stress-test scenarios:

- mixed corruption
- biased range corruption
- adversarial mod-6-passing noise

In [ ]:
def simple_sieve(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=int)
    s = np.ones(n + 1, dtype=bool)
    s[:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if s[i]:
            s[i*i:n+1:i] = False
    return np.nonzero(s)[0]

reference_primes = simple_sieve(N_MAX)
prime_set = set(reference_primes.tolist())
universe = np.arange(2, N_MAX + 1)
composites = np.array([n for n in universe if n not in prime_set], dtype=int)
mod6_composites = composites[np.isin(composites % 6, [1, 5])]

def make_observation(cfg: dict) -> dict:
    name = cfg["name"]
    kind = cfg["kind"]
    keep_fraction = cfg["keep_fraction"]
    noise_fraction = cfg["noise_fraction"]

    keep_count = int(round(keep_fraction * len(reference_primes)))
    noise_count = int(round(noise_fraction * len(reference_primes)))

    if kind == "biased_range":
        midpoint = N_MAX // 2
        low_primes = reference_primes[reference_primes <= midpoint]
        high_primes = reference_primes[reference_primes > midpoint]

        low_keep_count = min(len(low_primes), int(round(0.90 * len(low_primes))))
        remaining_keep = max(0, keep_count - low_keep_count)
        high_keep_count = min(len(high_primes), remaining_keep)

        kept_low = rng.choice(low_primes, size=low_keep_count, replace=False)
        kept_high = rng.choice(high_primes, size=high_keep_count, replace=False) if high_keep_count else np.array([], dtype=int)

        high_composites = composites[composites > midpoint]
        noise_values = rng.choice(high_composites, size=min(noise_count, len(high_composites)), replace=False)

        values = np.sort(np.unique(np.concatenate([kept_low, kept_high, noise_values])))

    elif kind == "adversarial_mod6":
        kept_primes = rng.choice(reference_primes, size=keep_count, replace=False)
        noise_values = rng.choice(mod6_composites, size=min(noise_count, len(mod6_composites)), replace=False)
        values = np.sort(np.unique(np.concatenate([kept_primes, noise_values])))

    else:
        kept_primes = rng.choice(reference_primes, size=keep_count, replace=False)
        noise_values = rng.choice(composites, size=min(noise_count, len(composites)), replace=False)
        values = np.sort(np.unique(np.concatenate([kept_primes, noise_values])))

    return {
        "name": name,
        "kind": kind,
        "keep_fraction": keep_fraction,
        "noise_fraction": noise_fraction,
        "values": values,
    }

observations = [make_observation(cfg) for cfg in SCENARIOS]

summary = {
    "n_max": int(N_MAX),
    "prime_count": int(len(reference_primes)),
    "composite_count": int(len(composites)),
    "mod6_composite_count": int(len(mod6_composites)),
    "scenario_count": int(len(observations)),
}

summary, [(o["name"], len(o["values"])) for o in observations]

## 4. Filters and scoring functions

In [ ]:
def residue_filter(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=int)
    keep = (values == 2) | (values == 3) | np.isin(values % 6, [1, 5])
    return np.sort(values[keep])

def passes_sieve(values: np.ndarray, q_max: int) -> np.ndarray:
    values = np.asarray(values, dtype=int)
    keep = np.ones(len(values), dtype=bool)
    filter_primes = reference_primes[reference_primes <= q_max]

    for q in filter_primes:
        keep &= ((values == q) | (values % q != 0))

    return keep

def sieve_filter(values: np.ndarray, q_max: int) -> np.ndarray:
    return np.sort(values[passes_sieve(values, q_max)])

def pi_model(x: np.ndarray | float) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    safe = np.maximum(x, 3.0)
    denom = np.log(safe) - 1.0
    denom = np.maximum(denom, 1.0)
    return safe / denom

def candidate_pool(q_max: int) -> np.ndarray:
    candidates = residue_filter(universe)
    return sieve_filter(candidates, q_max=q_max)

POOL = candidate_pool(Q_FILTER)

def density_drift(values: np.ndarray, scales: np.ndarray) -> float:
    values = np.sort(values)
    drifts = []
    for x in scales:
        obs_count = int(np.searchsorted(values, x, side="right"))
        prime_count = int(np.searchsorted(reference_primes, x, side="right"))
        drift = abs(obs_count - prime_count) / prime_count if prime_count else 0.0
        drifts.append(drift)
    return float(np.mean(drifts))

def gap_drift(values: np.ndarray) -> float:
    values = np.sort(np.unique(values.astype(int)))
    if len(values) < 3:
        return float("nan")
    gaps = np.diff(values)
    anchors = values[:-1]
    expected = np.maximum(np.log(np.maximum(anchors, 3)), 1.0)
    rel = np.abs(gaps - expected) / expected
    return float(np.mean(np.clip(rel, 0, 20)))

def gap_residual_stats(values: np.ndarray) -> dict:
    values = np.sort(np.unique(values.astype(int)))
    if len(values) < 3:
        return {
            "mean_gap_residual": float("nan"),
            "std_gap_residual": float("nan"),
            "median_abs_gap_residual": float("nan"),
        }
    gaps = np.diff(values)
    anchors = values[:-1]
    expected = np.maximum(np.log(np.maximum(anchors, 3)), 1.0)
    residuals = gaps - expected
    return {
        "mean_gap_residual": float(np.mean(residuals)),
        "std_gap_residual": float(np.std(residuals)),
        "median_abs_gap_residual": float(np.median(np.abs(residuals))),
    }

def metrics(values: np.ndarray, scenario: str, kind: str, method: str, scales: np.ndarray) -> dict:
    values = np.sort(np.unique(values.astype(int)))
    value_set = set(values.tolist())

    tp = len(value_set & prime_set)
    fp = len(value_set - prime_set)
    fn = len(prime_set - value_set)

    recovery = tp / len(reference_primes)
    precision = tp / len(values) if len(values) else float("nan")
    f1 = 2 * precision * recovery / (precision + recovery) if (precision + recovery) else float("nan")

    gap_stats = gap_residual_stats(values)

    return {
        "scenario": scenario,
        "kind": kind,
        "method": method,
        "count": int(len(values)),
        "true_positive": int(tp),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "precision": float(precision),
        "recovery": float(recovery),
        "f1": float(f1),
        "density_drift": density_drift(values, scales),
        "gap_drift": gap_drift(values),
        **gap_stats,
    }

def nearest_gap_scores(candidates: np.ndarray, current_values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    candidates = np.asarray(candidates, dtype=int)
    current_values = np.sort(np.unique(current_values.astype(int)))

    if len(current_values) == 0:
        return np.ones(len(candidates)), np.ones(len(candidates))

    pos = np.searchsorted(current_values, candidates)
    left_neighbor = np.where(pos > 0, current_values[np.maximum(pos - 1, 0)], -10**12)
    right_neighbor = np.where(pos < len(current_values), current_values[np.minimum(pos, len(current_values)-1)], 10**12)

    left_gap = candidates - left_neighbor
    right_gap = right_neighbor - candidates

    expected = np.maximum(np.log(np.maximum(candidates, 3)), 1.0)

    # Local score rewards splitting large gaps into plausible pieces near log(x).
    finite_left = left_neighbor > 0
    finite_right = right_neighbor < 10**11

    left_score = np.where(finite_left, np.exp(-np.abs(left_gap - expected) / expected), 0.5)
    right_score = np.where(finite_right, np.exp(-np.abs(right_gap - expected) / expected), 0.5)

    gap_score = 0.5 * (left_score + right_score)

    # Neighbor score rewards candidates not too close to existing points.
    nearest = np.minimum(np.where(finite_left, left_gap, expected), np.where(finite_right, right_gap, expected))
    neighbor_score = 1.0 - np.exp(-nearest / expected)

    return gap_score, neighbor_score

scales = np.unique(np.logspace(2, np.log10(N_MAX), 70).astype(int))

print("Functions ready")

## 5. Reconstruction methods

### Density-only

Replicates Notebook 07 style: fill each bin according to density need.

### Local-gap reconstruction

Uses:

- bin density need
- local expected gap \(\log x\)
- nearest-neighbor spacing consistency

In [ ]:
def density_only_completion(filtered_values: np.ndarray, target_count: int) -> tuple[np.ndarray, pd.DataFrame]:
    filtered_values = np.sort(np.unique(filtered_values.astype(int)))
    existing = set(filtered_values.tolist())
    available = np.array([v for v in POOL if v not in existing], dtype=int)

    needed_total = max(0, int(target_count - len(filtered_values)))
    if needed_total == 0 or len(available) == 0:
        return filtered_values, pd.DataFrame(columns=["candidate", "method", "bin_left", "bin_right", "score"])

    edges = np.unique(np.linspace(2, N_MAX, COMPLETION_BIN_COUNT + 1).astype(int))
    rows = []

    for left, right in zip(edges[:-1], edges[1:]):
        bin_available = available[(available >= left) & (available <= right)]
        if len(bin_available) == 0:
            continue

        observed_count = int(np.sum((filtered_values >= left) & (filtered_values <= right)))
        model_bin_count = int(max(0, round(pi_model(right) - pi_model(left))))
        bin_need = max(0, model_bin_count - observed_count)

        if bin_need == 0:
            continue

        center = 0.5 * (left + right)
        scale = max(1.0, right - left)
        centered_score = 1.0 - np.abs(bin_available - center) / scale
        jitter = rng.normal(0, 1e-6, size=len(bin_available))
        scores = centered_score + jitter

        order = np.argsort(scores)[::-1]
        chosen = bin_available[order[:min(bin_need, len(bin_available))]]
        score_lookup = dict(zip(bin_available.tolist(), scores.tolist()))

        for c in chosen:
            rows.append({
                "candidate": int(c),
                "method": "density_only",
                "bin_left": int(left),
                "bin_right": int(right),
                "score": float(score_lookup[int(c)]),
            })

    df = pd.DataFrame(rows)
    if len(df) > 0:
        df = df.sort_values("score", ascending=False).head(needed_total)
        selected = df["candidate"].to_numpy(dtype=int)
    else:
        selected = np.array([], dtype=int)

    reconstructed = np.sort(np.unique(np.concatenate([filtered_values, selected])))
    return reconstructed, df

def local_gap_completion(filtered_values: np.ndarray, target_count: int) -> tuple[np.ndarray, pd.DataFrame]:
    current = np.sort(np.unique(filtered_values.astype(int)))
    existing = set(current.tolist())
    available = np.array([v for v in POOL if v not in existing], dtype=int)

    needed_total = max(0, int(target_count - len(current)))
    if needed_total == 0 or len(available) == 0:
        return current, pd.DataFrame(columns=["candidate", "method", "bin_left", "bin_right", "score", "gap_score", "neighbor_score"])

    edges = np.unique(np.linspace(2, N_MAX, COMPLETION_BIN_COUNT + 1).astype(int))
    rows = []

    for left, right in zip(edges[:-1], edges[1:]):
        bin_available = available[(available >= left) & (available <= right)]
        if len(bin_available) == 0:
            continue

        observed_count = int(np.sum((current >= left) & (current <= right)))
        model_bin_count = int(max(0, round(pi_model(right) - pi_model(left))))
        bin_need = max(0, model_bin_count - observed_count)

        if bin_need == 0:
            continue

        center = 0.5 * (left + right)
        scale = max(1.0, right - left)
        density_score = 1.0 - np.abs(bin_available - center) / scale

        gap_score, neighbor_score = nearest_gap_scores(bin_available, current)

        jitter = rng.normal(0, 1e-6, size=len(bin_available))
        score = (
            W_DENSITY * density_score
            + W_GAP * gap_score
            + W_NEIGHBOR * neighbor_score
            + jitter
        )

        order = np.argsort(score)[::-1]
        chosen = bin_available[order[:min(bin_need, len(bin_available))]]

        lookup = {
            int(v): (float(score[i]), float(gap_score[i]), float(neighbor_score[i]), float(density_score[i]))
            for i, v in enumerate(bin_available)
        }

        for c in chosen:
            s, gs, ns, ds = lookup[int(c)]
            rows.append({
                "candidate": int(c),
                "method": "local_gap",
                "bin_left": int(left),
                "bin_right": int(right),
                "score": s,
                "gap_score": gs,
                "neighbor_score": ns,
                "density_score": ds,
            })

    df = pd.DataFrame(rows)
    if len(df) > 0:
        df = df.sort_values("score", ascending=False).head(needed_total)
        selected = df["candidate"].to_numpy(dtype=int)
    else:
        selected = np.array([], dtype=int)

    reconstructed = np.sort(np.unique(np.concatenate([current, selected])))
    return reconstructed, df

print("Reconstruction methods ready")

## 6. Run reconstruction comparison

For each scenario:

1. raw observation
2. sieve-cleaned observation
3. density-only reconstruction
4. local-gap reconstruction

In [ ]:
target_count = len(reference_primes)

method_metrics = []
completion_frames = []
reconstructed_sets = {}

for obs in observations:
    scenario = obs["name"]
    kind = obs["kind"]

    raw = np.sort(obs["values"])
    cleaned = sieve_filter(residue_filter(raw), q_max=Q_FILTER)

    density_recon, density_completions = density_only_completion(cleaned, target_count=target_count)
    gap_recon, gap_completions = local_gap_completion(cleaned, target_count=target_count)

    reconstructed_sets[(scenario, "raw")] = raw
    reconstructed_sets[(scenario, "sieve_cleaned")] = cleaned
    reconstructed_sets[(scenario, "density_only")] = density_recon
    reconstructed_sets[(scenario, "local_gap")] = gap_recon

    method_metrics.append(metrics(raw, scenario, kind, "raw", scales))
    method_metrics.append(metrics(cleaned, scenario, kind, "sieve_cleaned", scales))
    method_metrics.append(metrics(density_recon, scenario, kind, "density_only", scales))
    method_metrics.append(metrics(gap_recon, scenario, kind, "local_gap", scales))

    density_completions["scenario"] = scenario
    density_completions["kind"] = kind
    gap_completions["scenario"] = scenario
    gap_completions["kind"] = kind

    completion_frames.append(density_completions)
    completion_frames.append(gap_completions)

method_metrics_df = pd.DataFrame(method_metrics)
candidate_completions_df = pd.concat(completion_frames, ignore_index=True) if completion_frames else pd.DataFrame()

method_metrics_df.head(12), candidate_completions_df.head()

## 7. Method deltas

Measure whether local-gap reconstruction improves structural metrics relative to density-only reconstruction.

In [ ]:
delta_rows = []

for scenario in [cfg["name"] for cfg in SCENARIOS]:
    d = method_metrics_df[(method_metrics_df["scenario"] == scenario) & (method_metrics_df["method"] == "density_only")].iloc[0]
    g = method_metrics_df[(method_metrics_df["scenario"] == scenario) & (method_metrics_df["method"] == "local_gap")].iloc[0]

    delta_rows.append({
        "scenario": scenario,
        "kind": g["kind"],
        "delta_precision_local_minus_density": float(g["precision"] - d["precision"]),
        "delta_recovery_local_minus_density": float(g["recovery"] - d["recovery"]),
        "delta_f1_local_minus_density": float(g["f1"] - d["f1"]),
        "delta_density_drift_local_minus_density": float(g["density_drift"] - d["density_drift"]),
        "delta_gap_drift_local_minus_density": float(g["gap_drift"] - d["gap_drift"]),
        "density_gap_drift": float(d["gap_drift"]),
        "local_gap_drift": float(g["gap_drift"]),
    })

method_deltas_df = pd.DataFrame(delta_rows)

measurement = {
    "mean_density_only_f1": float(method_metrics_df[method_metrics_df["method"] == "density_only"]["f1"].mean()),
    "mean_local_gap_f1": float(method_metrics_df[method_metrics_df["method"] == "local_gap"]["f1"].mean()),
    "mean_density_only_gap_drift": float(method_metrics_df[method_metrics_df["method"] == "density_only"]["gap_drift"].mean()),
    "mean_local_gap_drift": float(method_metrics_df[method_metrics_df["method"] == "local_gap"]["gap_drift"].mean()),
    "mean_gap_drift_delta_local_minus_density": float(method_deltas_df["delta_gap_drift_local_minus_density"].mean()),
    "candidate_completion_count": int(len(candidate_completions_df)),
}

cgcs = {
    "score": measurement["mean_local_gap_f1"],
    "definition": "mean F1 score of local-gap reconstruction across stress-test scenarios",
    "interpretation": "Local gap reconstruction preserves global recovery while testing local spacing realism.",
}

measurement, method_deltas_df

## 8. Figure 1 — method precision and recovery

Compare raw, sieve-cleaned, density-only, and local-gap stages.

In [ ]:
method_order = ["raw", "sieve_cleaned", "density_only", "local_gap"]
scenario_order = [cfg["name"] for cfg in SCENARIOS]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(method_order))

for scenario in scenario_order:
    sub = method_metrics_df[method_metrics_df["scenario"] == scenario].set_index("method").loc[method_order]
    ax.plot(x, sub["precision"], marker="o", label=f"{scenario} precision")
    ax.plot(x, sub["recovery"], marker="x", linestyle="--", label=f"{scenario} recovery")

ax.set_xticks(x)
ax.set_xticklabels(method_order, rotation=20)
ax.set_ylim(0, 1.05)
ax.set_title("Method precision and recovery")
ax.set_xlabel("method")
ax.set_ylabel("score")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_method_precision_recovery.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

## 9. Figure 2 — F1 by method

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for scenario in scenario_order:
    sub = method_metrics_df[method_metrics_df["scenario"] == scenario].set_index("method").loc[method_order]
    ax.plot(method_order, sub["f1"], marker="o", label=scenario)

ax.set_ylim(0, 1.05)
ax.set_title("F1 by reconstruction method")
ax.set_xlabel("method")
ax.set_ylabel("F1")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_f1_by_method.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

## 10. Figure 3 — gap drift by method

Lower is better.  
This is the key Notebook 08 diagnostic.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for scenario in scenario_order:
    sub = method_metrics_df[method_metrics_df["scenario"] == scenario].set_index("method").loc[method_order]
    ax.plot(method_order, sub["gap_drift"], marker="o", label=scenario)

ax.set_title("Gap drift by reconstruction method")
ax.set_xlabel("method")
ax.set_ylabel("mean relative gap drift")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_gap_drift_by_method.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

fig3_path

## 11. Figure 4 — density drift by method

Density reconstruction should preserve count-scale structure.  
Local-gap reconstruction should not break it.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for scenario in scenario_order:
    sub = method_metrics_df[method_metrics_df["scenario"] == scenario].set_index("method").loc[method_order]
    ax.plot(method_order, sub["density_drift"], marker="o", label=scenario)

ax.set_title("Density drift by reconstruction method")
ax.set_xlabel("method")
ax.set_ylabel("density drift")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig4_path = FIG_DIR / f"{NOTEBOOK_NUM}_density_drift_by_method.png"
fig.savefig(fig4_path, dpi=180, bbox_inches="tight")
plt.show()

fig4_path

## 12. Figure 5 — density-only vs local-gap deltas

Negative gap-drift delta means local-gap reconstruction improved gap structure.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
labels = method_deltas_df["scenario"].str.replace("_", "\n")

ax.bar(labels, method_deltas_df["delta_gap_drift_local_minus_density"])
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("Local-gap minus density-only gap drift")
ax.set_xlabel("scenario")
ax.set_ylabel("delta gap drift")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, axis="y", alpha=0.3)

fig5_path = FIG_DIR / f"{NOTEBOOK_NUM}_gap_drift_delta.png"
fig.savefig(fig5_path, dpi=180, bbox_inches="tight")
plt.show()

fig5_path

## 13. Figure 6 — candidate completion distribution

Compare density-only and local-gap completions across scale.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for method in ["density_only", "local_gap"]:
    sub = candidate_completions_df[candidate_completions_df["method"] == method]
    if len(sub) > 0:
        ax.hist(sub["candidate"], bins=50, alpha=0.45, label=method)

ax.set_title("Candidate completion distribution by method")
ax.set_xlabel("candidate value")
ax.set_ylabel("frequency")
ax.legend()
ax.grid(True, alpha=0.3)

fig6_path = FIG_DIR / f"{NOTEBOOK_NUM}_candidate_completion_distribution_by_method.png"
fig.savefig(fig6_path, dpi=180, bbox_inches="tight")
plt.show()

fig6_path

## 14. Figure 7 — reconstructed gaps vs expected \(\log x\)

Use the hardest adversarial case as representative.

In [ ]:
representative = "adversarial_mod6_noise_100"

fig, ax = plt.subplots(figsize=(11, 5))

for method in ["density_only", "local_gap"]:
    vals = reconstructed_sets[(representative, method)]
    vals = np.sort(np.unique(vals))
    gaps = np.diff(vals)
    anchors = vals[:-1]
    sample_idx = np.linspace(0, len(gaps)-1, min(500, len(gaps))).astype(int)
    ax.scatter(anchors[sample_idx], gaps[sample_idx], s=10, alpha=0.35, label=f"{method} gaps")

expected = np.log(np.maximum(anchors[sample_idx], 3))
ax.plot(anchors[sample_idx], expected, linewidth=2, label="expected log x")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Reconstructed gaps versus expected log x")
ax.set_xlabel("x")
ax.set_ylabel("gap")
ax.legend()
ax.grid(True, alpha=0.3)

fig7_path = FIG_DIR / f"{NOTEBOOK_NUM}_reconstructed_gap_vs_logx.png"
fig.savefig(fig7_path, dpi=180, bbox_inches="tight")
plt.show()

fig7_path

## 15. Figure 8 — gap residual distributions

Compare gap residuals:

\[
r_i = g_i - \log p_i
\]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for method in ["density_only", "local_gap"]:
    vals = reconstructed_sets[(representative, method)]
    vals = np.sort(np.unique(vals))
    gaps = np.diff(vals)
    anchors = vals[:-1]
    residuals = gaps - np.log(np.maximum(anchors, 3))
    residuals = residuals[np.abs(residuals) <= 100]
    ax.hist(residuals, bins=80, alpha=0.45, label=method)

ax.axvline(0, linestyle="--", linewidth=1)
ax.set_title("Gap residual distribution")
ax.set_xlabel("gap - log(x)")
ax.set_ylabel("frequency")
ax.legend()
ax.grid(True, alpha=0.3)

fig8_path = FIG_DIR / f"{NOTEBOOK_NUM}_gap_residual_distribution.png"
fig.savefig(fig8_path, dpi=180, bbox_inches="tight")
plt.show()

fig8_path

## 16. Interpretation

1. **Density-only reconstruction** repairs global count.

2. **Local-gap reconstruction** adds a local spacing constraint.

3. **Gap drift** checks whether reconstructed candidates are structurally realistic.

4. **F1 alone is not enough** because a reconstruction can match prime count while missing local gap structure.

Core statement:

> Density restores missing mass; local gap scoring tests whether reconstructed mass lands in structurally plausible places.

In [ ]:
interpretation_lines = [
    f"# {NOTEBOOK_TITLE}",
    "",
    "## Constraint result",
    "",
    "This notebook compares density-only reconstruction against local-gap reconstruction.",
    "",
    "Notebook 07 showed that density reconstruction can repair global count while still leaving local spacing artifacts.",
    "",
    "Notebook 08 adds local gap scoring to test whether reconstructed candidates match expected spacing.",
    "",
    "## Main findings",
    "",
    "Density-only reconstruction restores missing mass and improves recovery.",
    "",
    "Local-gap reconstruction preserves the global reconstruction objective while adding a local spacing test.",
    "",
    "F1 alone is not sufficient because a reconstruction can score well on identity/count metrics while retaining unrealistic gap structure.",
    "",
    "## Summary metrics",
    "",
    f"- mean density-only F1 = {measurement['mean_density_only_f1']:.6f}",
    f"- mean local-gap F1 = {measurement['mean_local_gap_f1']:.6f}",
    f"- mean density-only gap drift = {measurement['mean_density_only_gap_drift']:.6f}",
    f"- mean local-gap gap drift = {measurement['mean_local_gap_drift']:.6f}",
    f"- mean gap-drift delta local-minus-density = {measurement['mean_gap_drift_delta_local_minus_density']:.6f}",
    f"- candidate completion count = {measurement['candidate_completion_count']}",
    "",
    "## Interpretation",
    "",
    "Sieve filtering restores precision by removing composite noise.",
    "",
    "Density reconstruction restores global missing mass.",
    "",
    "Local-gap reconstruction tests whether added candidates are placed in structurally plausible gaps.",
    "",
    "## Caution",
    "",
    "Local gap scoring improves structural diagnostics, but candidate completions remain hypotheses rather than certified primes.",
    "",
    "## Core statement",
    "",
    "Density restores missing mass; local gap scoring tests whether reconstructed mass lands in structurally plausible places.",
]

interpretation = "\n".join(interpretation_lines)

figure_paths = [fig1_path, fig2_path, fig3_path, fig4_path, fig5_path, fig6_path, fig7_path, fig8_path]
figure_titles = [
    "Method precision and recovery",
    "F1 by method",
    "Gap drift by method",
    "Density drift by method",
    "Gap drift delta",
    "Candidate completion distribution",
    "Reconstructed gaps versus expected log x",
    "Gap residual distribution",
]

figures_md = "\n\n## Figures\n\n"
for i, (fig, title) in enumerate(zip(figure_paths, figure_titles), start=1):
    figures_md += f"### Figure {i} — {title}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

print(interpretation + figures_md)

## 17. Export data, docs, math, and TeX

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
method_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_method_metrics.csv"
method_deltas_path = DATA_DIR / f"{NOTEBOOK_NUM}_method_deltas.csv"
candidate_completions_path = DATA_DIR / f"{NOTEBOOK_NUM}_candidate_completions.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
method_metrics_df.to_csv(method_metrics_path, index=False)
method_deltas_df.to_csv(method_deltas_path, index=False)
candidate_completions_df.to_csv(candidate_completions_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "method_metrics": str(method_metrics_path),
        "method_deltas": str(method_deltas_path),
        "candidate_completions": str(candidate_completions_path),
    },
    "docs": {
        "interpretation": str(interpretation_path),
        "design_notes": str(design_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
interpretation_path.write_text(interpretation + figures_md + "\n", encoding="utf-8")

design_lines = [
    f"# Design Notes — {NOTEBOOK_TITLE}",
    "",
    "## Notebook role",
    "",
    "Notebook 08 follows Notebook 07 by improving candidate completion with local gap scoring.",
    "",
    "Notebook 07 showed that density-only reconstruction repairs global count but can leave local spacing artifacts.",
    "",
    "## Compared methods",
    "",
    "1. raw observation",
    "2. sieve-cleaned observation",
    "3. density-only reconstruction",
    "4. local-gap reconstruction",
    "",
    "## Measurements",
    "",
    "1. precision",
    "2. recovery",
    "3. F1",
    "4. density drift",
    "5. gap drift",
    "6. gap residuals",
    "",
    "## Core claim",
    "",
    "Density restores missing mass; local gap scoring tests whether reconstructed mass lands in structurally plausible places.",
    "",
    "## Caution",
    "",
    "Candidate completions are hypotheses, not certified primes.",
    "",
    "## Handoff",
    "",
    "Notebook 09 should test windowed/local reconstruction and compare reconstruction stability across sliding intervals.",
]

design_path.write_text("\n".join(design_lines) + "\n", encoding="utf-8")

summary_tex_lines = [
    rf"\section*{{{NOTEBOOK_TITLE}}}",
    "",
    r"This notebook compares density-only reconstruction against local-gap reconstruction.",
    "",
    r"Expected local prime spacing is approximated by",
    r"\[",
    r"g(x)\sim \log x.",
    r"\]",
    "",
    r"Gap drift is measured by",
    r"\[",
    r"drift_{\mathrm{gap}} =",
    r"\operatorname{mean}_i \frac{|g_i-\log p_i|}{\log p_i}.",
    r"\]",
    "",
    rf"For $N={N_MAX:,}$:",
    r"\begin{itemize}",
    rf"  \item mean density-only $F_1 = {measurement['mean_density_only_f1']:.6f}$",
    rf"  \item mean local-gap $F_1 = {measurement['mean_local_gap_f1']:.6f}$",
    rf"  \item mean density-only gap drift $= {measurement['mean_density_only_gap_drift']:.6f}$",
    rf"  \item mean local-gap gap drift $= {measurement['mean_local_gap_drift']:.6f}$",
    r"\end{itemize}",
    "",
    r"Density restores missing mass; local gap scoring tests structural plausibility.",
]

summary_tex_path.write_text("\n".join(summary_tex_lines) + "\n", encoding="utf-8")

math_tex_lines = [
    r"\documentclass{article}",
    r"\usepackage{amsmath}",
    r"\usepackage{amssymb}",
    r"\usepackage[margin=1in]{geometry}",
    "",
    r"\begin{document}",
    "",
    r"\section*{Math Notes: Local Gap Reconstruction}",
    "",
    r"\subsection*{Prime set}",
    r"\[P_N=\{p:p\le N\}.\]",
    "",
    r"\subsection*{Precision and recovery}",
    r"\[precision=\frac{|\hat{P}_N\cap P_N|}{|\hat{P}_N|}.\]",
    r"\[recovery=\frac{|\hat{P}_N\cap P_N|}{|P_N|}.\]",
    "",
    r"\subsection*{F1}",
    r"\[F_1=\frac{2\cdot precision\cdot recovery}{precision+recovery}.\]",
    "",
    r"\subsection*{Expected local gap}",
    r"\[g(x)\sim \log x.\]",
    "",
    r"\subsection*{Candidate score}",
    r"\[",
    r"score(x)=w_d\,score_{\mathrm{density}}(x)+w_g\,score_{\mathrm{gap}}(x)+w_n\,score_{\mathrm{neighbor}}(x).",
    r"\]",
    "",
    r"\subsection*{Gap drift}",
    r"\[",
    r"drift_{\mathrm{gap}}=",
    r"\operatorname{mean}_i \frac{|g_i-\log p_i|}{\log p_i}.",
    r"\]",
    "",
    r"Candidate completions are hypotheses, not proofs of primality.",
    "",
    r"\end{document}",
]

math_tex_path.write_text("\n".join(math_tex_lines) + "\n", encoding="utf-8")

summary_path, method_metrics_path, method_deltas_path, candidate_completions_path, metadata_path, interpretation_path, design_path, summary_tex_path, math_tex_path

## 18. Export zip

Pi-stage-lab style root export zip, with optional Colab download lines left commented.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 19. Next notebook handoff

Next notebook:

```text
09_windowed_reconstruction_stability.ipynb
```

Purpose:

> test reconstruction stability across sliding intervals, so global density does not hide local failures.

In [ ]:
next_step = "Notebook 09: windowed reconstruction stability."
print(next_step)